> Experiments with [model.signal](../trapallada/model/signal.py)  module

In [1]:
import pathlib
import pprint

import yaml
# import numpy as np
import pandas as pd
from IPython.display import display

# import jax
# import jax.numpy as jnp
import jax.random as jr

import torch

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


# TODO

- normalize the signals so that they are between 0 and 1, and then `tanh` en el codificador

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()
input_directory

PosixPath('/home/manu/Sync/git/trapallada/data')

In [4]:
base_filename = '4_classes_modulated'

# signals_file = input_directory / f'{base_filename}_signals.pt'
signals_file = input_directory / '20_600_signals.pt'
assert signals_file.exists()

# metadata_file = input_directory / f'{base_filename}_labels.parquet'
metadata_file = input_directory / f'20_600_metadata.parquet'
assert metadata_file.exists

In [5]:
signals = torch.load(signals_file)
torch.isnan(signals).sum()

tensor(0)

In [6]:
(signals[:, 0, :].sum(axis=1) == 0.).sum()

tensor(257)

In [7]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=False)
len(ds)

35666

In [8]:
# parameters_file = pathlib.Path.cwd().parent / 'parameters_vae_on_signals.yaml'
parameters_file = pathlib.Path.cwd().parent / 'experiments' / 'new' / 'real_z_50_300_epochs.yaml'
assert parameters_file.exists()

In [9]:
with parameters_file.open() as f:
    parameters = yaml.safe_load(f)

In [10]:
# pprint.pprint(parameters)
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
encoder:
  input_size:
  - 1
  - 600
loss:
  x_sd: 0.25
model:
  channels:
  - 12
  - 12
  - 12
  dim_z: 20 
  kernel_sizes:
  - 128
  - 64
  - 32
  stride: 2
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 300
  train_test_split:
    test_size: 0.2


In [11]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])


In [12]:
batch = next(iter(train_dl))
len(batch)

3

In [13]:
batch[0]

tensor([13347,  1525,  5918, 31766, 27932,  5615, 22135, 12954,  7686,  4536,
        34711, 15934, 21067, 20616, 27934, 13844, 22228, 26299, 32741, 16329,
         7505, 22743, 16190, 30060, 34877, 25032,  2324, 19167, 30460,   560,
        23694,  8547])

In [14]:
signals_len = ds[0][1].shape[-1]
signals_len

600

In [15]:
tuple(ds[0][1].shape)

(1, 600)

Input size must be adjusted

In [16]:
parameters['encoder']['input_size'] = list(ds[0][1].shape)

In [17]:
x = batch[1][0].numpy()
encoder = trapallada.model.signal.Encoder(
    **(parameters['model'] | parameters['encoder']),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
encoder.debug_call(x)
print(encoder)

after equinox.nn._conv: (12, 237)
after jax._src.nn.functions: (12, 237)
after equinox.nn._normalisation: (12, 237)
after equinox.nn._conv: (12, 87)
after jax._src.nn.functions: (12, 87)
after equinox.nn._normalisation: (12, 87)
after equinox.nn._conv: (12, 28)
after jax._src.nn.functions: (12, 28)
after equinox.nn._normalisation: (12, 28)
after jax._src.numpy.lax_numpy: (336,)
after equinox.nn._linear: (40,)
after trapallada.model.signal: (40,)
Encoder(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,1,128],
      bias=f32[12,1],
      in_channels=1,
      out_channels=12,
      kernel_size=(128,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True,
      padding_mode='ZEROS'
    ),
    <wrapped function relu>,
    LayerNorm(
      shape=(12, 237),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      weight=f32[12,237],
      bias=f32[12,237]
    ),
    Conv1d(
      num_spatial_dims=1,
      weight=f

In [18]:
encoder.layers_spatial_sizes

[600, 237, 87, 28, 336]

In [19]:
z = encoder(x)
z.shape

(40,)

In [20]:
z_mean = jr.normal(jr.PRNGKey(42), (parameters['model']['dim_z'],))
decoder = trapallada.model.signal.Decoder(
    **(parameters['model'] | {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
decoder.debug_call(z_mean)

after equinox.nn._linear: (336,)
after jax._src.nn.functions: (336,)
after trapallada.model.signal: (12, 28)
after equinox.nn._conv: (12, 87)
after jax._src.nn.functions: (12, 87)
after equinox.nn._normalisation: (12, 87)
after equinox.nn._conv: (12, 237)
after jax._src.nn.functions: (12, 237)
after equinox.nn._normalisation: (12, 237)
after equinox.nn._conv: (1, 600)


In [21]:
vae = trapallada.model.signal.VAE(encoder, decoder)
x_pred, z_mean, z_sd = vae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

((1, 600), (20,), (20,))

In [22]:
x_pred

Array([[-3.49293873e-02, -4.04043123e-02, -4.43984494e-02,
        -8.32946897e-02, -9.00559500e-02, -5.72892353e-02,
        -1.22930884e-01, -3.86460721e-02, -1.12945139e-01,
        -1.89644899e-02, -9.80305523e-02,  3.84300388e-03,
        -6.46368116e-02,  2.57855039e-02, -1.04343280e-01,
         1.42081734e-02, -1.14371352e-01,  7.91737065e-02,
        -1.65538922e-01,  7.83076137e-02, -9.72048789e-02,
         6.38219640e-02, -9.84279886e-02,  5.39849848e-02,
        -1.63109243e-01,  1.76710561e-01, -1.62267268e-01,
         3.63025591e-02, -1.23416960e-01, -2.69644987e-02,
        -2.30698138e-01,  1.93829779e-02, -9.13436264e-02,
         1.04522102e-01, -2.39878997e-01, -6.75275922e-03,
        -2.50184238e-01,  1.05357338e-02, -2.70644784e-01,
        -3.71328890e-02, -1.98457748e-01,  4.78794947e-02,
        -2.85348088e-01, -1.60263062e-01,  4.76111658e-03,
         6.46916106e-02, -2.47874096e-01, -7.94915296e-03,
        -3.64678472e-01, -1.12453014e-01, -2.25765347e-0

In [23]:
# torch.isnan(ds[:][1]).sum()

In [24]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])
for i_batch, (i_sample, x, _) in enumerate(train_dl):

    print(torch.isnan(x).sum())

tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)
tensor(0)


In [25]:
# model, results_df = trapallada.core.fit_vae_on_signals(train_dl, validation_dl, parameters)